<a href="https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/DETR/DETR_minimal_example_(with_DetrFeatureExtractor).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## DETR: inference notebook

`DetrFeatureExtractor` was renamed to `DetrImageProcessor`. This notebook is kept for existing Colab links and now uses the current API. Prefer [`DETR_minimal_example.ipynb`](https://github.com/NielsRogge/Transformers-Tutorials/blob/master/DETR/DETR_minimal_example.ipynb) for the full walkthrough, including decoder attention visualization.

In this notebook, we run [DETR](https://huggingface.co/docs/transformers/model_doc/detr) on a COCO validation image.

## Set-up environment

In [ ]:
%pip install -q transformers torch timm pillow matplotlib requests

## Prepare the image using `DetrImageProcessor`

In [ ]:
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
image

In [ ]:
from transformers import DetrImageProcessor, DetrForObjectDetection
import torch

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
encoding = processor(image, return_tensors="pt")
print(encoding["pixel_values"].shape)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
model.to(device)
model.eval()

with torch.no_grad():
    outputs = model(**{k: v.to(device) for k, v in encoding.items()})

In [ ]:
import matplotlib.pyplot as plt

# colors for visualization
COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]


def plot_results(pil_img, scores, labels, boxes, id2label):
    plt.figure(figsize=(16, 10))
    plt.imshow(pil_img)
    ax = plt.gca()
    colors = COLORS * 100
    for score, label, (xmin, ymin, xmax, ymax), c in zip(
        scores.tolist(), labels.tolist(), boxes.tolist(), colors
    ):
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=3))
        text = f"{id2label[label]}: {score:0.2f}"
        ax.text(xmin, ymin, text, fontsize=15,
                bbox=dict(facecolor="yellow", alpha=0.5))
    plt.axis("off")
    plt.show()


In [ ]:
width, height = image.size
results = processor.post_process_object_detection(
    outputs, target_sizes=[(height, width)], threshold=0.9
)[0]
plot_results(image, results["scores"].cpu(), results["labels"].cpu(), results["boxes"].cpu(), model.config.id2label)